# Aicademy 360

# Teacher sir shahzeb Malik

# ANN Salary Regression Assingment 1



In [ ]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
# load the customer churn dataset.

data = pd.read_csv('Churn_Modelling.csv')

In [3]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# EstimatedSalary is the value to predict.
X = data.drop(
    columns=["EstimatedSalary", "Exited", "RowNumber", "CustomerId", "Surname"]
)
y = data["EstimatedSalary"].astype("float32")

print("Input columns:", X.columns.tolist())
print("Target:", y.name)

Input columns: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember']
Target: EstimatedSalary


In [5]:
numeric_features = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember"
]
categorical_features = ["Geography", "Gender"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
X_train_scaled = preprocessor.fit_transform(X_train).astype("float32")
X_test_scaled = preprocessor.transform(X_test).astype("float32")
print(X_train_scaled.shape, X_test_scaled.shape)

(8000, 12) (2000, 12)


In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="linear")  # continuous numeric prediction
])
model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=10, restore_best_weights=True
        )
    ],
    verbose=1
)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 13306754048.0000 - mean_absolute_error: 99970.5234 - val_loss: 13687548928.0000 - val_mean_absolute_error: 101987.2188
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 13187345408.0000 - mean_absolute_error: 99372.6875 - val_loss: 13426369536.0000 - val_mean_absolute_error: 100709.7109
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 12721254400.0000 - mean_absolute_error: 97053.9062 - val_loss: 12705913856.0000 - val_mean_absolute_error: 97182.8828
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 11751748608.0000 - mean_absolute_error: 92207.0703 - val_loss: 11442049024.0000 - val_mean_absolute_error: 90962.5469
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 10280436736.0000 - mean_absolute_error: 84897.4922 - val_loss: 9720150016.0000 - val_mean_absolute_error: 82419.1641
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8477632000.0000 - mean_absolute_error: 7588

In [7]:
import tensorflow as tf

In [8]:
predictions = model.predict(X_test_scaled).ravel()
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
MAE:  49,869.79
RMSE: 57,568.54
R²:   -0.0039


In [9]:
model.save("salary_model.keras")
joblib.dump(preprocessor, "preprocessor.joblib")  

['preprocessor.joblib']

In [10]:
data.shape

(10000, 14)

In [11]:
data.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')